# Level 1 — Lesson 1: Before You Paste
## Shadow AI, Institutional Policy, and Patient Data

**Teaching time:** ~35–45 minutes  
**Core question:** *Before I ask an LLM for help, am I allowed to give it this information?*


In healthcare and research, the **first** question you should ask **before** interacting with AI is:

```text
Can I share this information with this AI system?
```

By the end of this lesson, you should be able to:

- explain **Shadow AI** in practical terms;
- understand why institutional policy takes precedence over convenience;
- recognize how sensitive information enters prompts accidentally;
- distinguish public data, confidential information, personal/patient data, and secrets;
- explain why pseudonymised data are not automatically anonymous;
- recognize that DICOM files contain more than voxels;
- rewrite a debugging request to share only what is necessary;
- explain why local/open-weight models are not automatically compliant or safe;
- know what to do if sensitive information is shared accidentally.

> **This is educational material, not legal advice. Your institution's policies, contracts, information-security rules, ethics approvals, and applicable law take precedence.**

# 0. The rule before the prompt

> **Technically possible does not mean institutionally permitted.**

A public chatbot may be accessible from your work computer. You may be able to paste a report, screenshot, source file, or error log into it.

That does **not** answer whether you are authorized to do so.

A useful sequence is:

```text
Is this tool approved?
        ↓
Is this type of information allowed?
        ↓
Can I solve the task with less information?
        ↓
Only then: write the prompt
```

# 1. What is Shadow AI?

For this course, **Shadow AI** means:

> **Using AI tools for organizational work outside the organization's approved governance, security, procurement, or IT processes.**

Examples might include:

- using a personal chatbot account for hospital work when only an institutional system is approved;
- pasting internal code into an unapproved external service;
- installing an AI coding extension without checking whether it is permitted;
- connecting an assistant to institutional cloud storage without authorization;
- using a locally downloaded model on a workstation that is not approved for the relevant data.

Shadow AI is not defined by whether a tool is *good* or *bad*.

It is about whether the use sits inside the organization's governance.

## Why does Shadow AI happen?

Usually for understandable reasons:

```text
"I just need to fix this error."
"The approved tool is slower."
"This isn't really patient data."
"I'm only pasting one screenshot."
"The model runs locally."
"Everyone else is already using it."
```

These statements may explain the temptation. They do not establish permission.

### Discussion

What makes Shadow AI particularly tempting in scientific coding?

What could an institution do to make **approved** AI use easier than unapproved use?

# 2. A practical data-classification heuristic

The following is a **teaching heuristic**, not a universal legal classification system.

### Public / intentionally shareable

```text
public documentation
public GitHub repository
Medical Segmentation Decathlon data
synthetic data
generic teaching error
```

### Internal / confidential

```text
unpublished manuscript
internal planning protocol
private source code
commercially confidential information
internal meeting notes
security architecture
```

### Personal / patient-related

```text
clinical note
local patient CT or MRI
RTPLAN / RTSTRUCT / RTDOSE
treatment dates
patient identifiers
rare clinical history
treatment-planning screenshots
```

### Credentials and secrets

```text
password
API key
access token
private key
database connection string
```

> **"Not patient data" does not automatically mean "safe to share."**

# 3. Health data receive special protection

Under the EU General Data Protection Regulation (GDPR), data concerning health are a **special category of personal data** and receive additional protection.

The GDPR uses a broad concept of health data: information revealing a person's past, current, or future physical or mental health status can qualify.

For this course, the practical lesson is:

> **Do not decide that health information is safe to upload merely because obvious identifiers were removed.**

Primary source:

- GDPR / Regulation (EU) 2016/679: https://eur-lex.europa.eu/eli/reg/2016/679/oj

## Pseudonymised is not the same as anonymous

Suppose:

```text
Jon Doe → Patient_0042
```

Replacing the name reduces direct identifiability.

But if `Patient_0042` can be **linked back** to Jon Doe using other information or a lookup table, the data are **pseudonymised**, not necessarily anonymous.

Conceptually:

```text
pseudonymisation
    reduces direct linkability
    but does not necessarily remove personal-data status

anonymisation
    aims to make the data no longer attributable to a person
```

> **"De-identified" is not a magic technical state. Ask what was removed, what remains, and whether re-identification is still reasonably possible.**

> **Identification may even be possible even if all text identifiers are removed! Modern face recognition software can for example idenitify faces from head CT scans.**

EDPB resources:

- https://www.edpb.europa.eu/topics/ai-and-technology/anonymisation-pseudonymisation_en
- https://www.edpb.europa.eu/our-work-tools/documents/public-consultations/2025/guidelines-012025-pseudonymisation_en

# 4. DICOM is not just the voxels

---



A common medical-imaging mistake is to think:

```text
image = voxel array
```

A DICOM object can contain many metadata fields in addition to pixel data, such as:

- patient name or ID;
- birth date;
- accession number;
- study and series dates;
- institution;
- referring or performing personnel;
- descriptions and free text;
- device information;
- UIDs that may allow linkage across systems or datasets.

Some images may also contain **burned-in text inside the pixels themselves**.

Let's inspect an entirely synthetic example.

In [ ]:
!pip -q install pydicom

In [ ]:
from pydicom.dataset import Dataset

ds = Dataset()

# Everything below is fictional teaching data.
ds.PatientName = "EXAMPLE^ALICE"
ds.PatientID = "PATIENT-0042"
ds.PatientBirthDate = "19700101"
ds.AccessionNumber = "ACC-123456"
ds.InstitutionName = "Example University Hospital"
ds.StudyDate = "20261015"
ds.SeriesDescription = "Planning CT"
ds.ReferringPhysicianName = "EXAMPLE^DR"
ds.BurnedInAnnotation = "YES"

print(ds)

### Exercise

Imagine you want help debugging Python code that reads this DICOM object.

Which fields does the LLM actually need?

A useful debugging request might need:

```text
DICOM object/modality
package version
shape / dtype
relevant tag keyword
exception type
small synthetic example
```

It probably does **not** need the fictional identity, birth date, institution, or accession number.

> **The safest useful prompt is often not a "cleaned" copy of the real data. It is a smaller synthetic example that reproduces the technical problem.**

# 5. Sensitive information often leaks through the edges

Consider this fictional traceback:

```text
Traceback (most recent call last):
  File "/hospital/research/HNC/Mueller_Anna_1944-03-02/RTSTRUCT.dcm", line ...
ValueError: expected ContourData to contain triplets
```

The technical signal is:

```text
ValueError: expected ContourData to contain triplets
```

But the path also contains identity and clinical context.

Other leakage routes include:

- screenshots and visible browser tabs;
- filenames and directory names;
- notebook output;
- terminal history;
- comments and logs;
- private repository URLs;
- copied email threads;
- environment variables;
- image metadata.

# 6. Rewrite the debugging request

### Unsafe version

> My code fails on `/hospital/research/HNC/Mueller_Anna_1944-03-02/RTSTRUCT.dcm` with `ValueError: expected ContourData to contain triplets`. Here is the full file and traceback. Fix it.

### Better version

> I am parsing an RTSTRUCT using Python and pydicom. For one contour, `ContourData` does not contain a number of values divisible by three, and my parser raises:
>
> ```text
> ValueError: expected ContourData to contain triplets
> ```
>
> What assumptions should I check in my parser? Please suggest a minimal debugging strategy before rewriting the code.

The second prompt contains more **technical signal** and less irrelevant information.

## Exercise — make the minimum useful prompt

You see this fictional error:

```text
File "/clinic/proton/patient_923881/plan_A/optimization.py", line 218
RuntimeError: The size of tensor a (128) must match the size of tensor b (127)
```

Write a new prompt that contains enough information to diagnose the shape mismatch **without** including the patient folder, institutional path, or data files.

Useful information might be:

```text
tensor A shape
tensor B shape
the operation that fails
the relevant 5–15 lines of generic code
PyTorch version if relevant
expected shape
```

# 7. Data minimisation is useful for debugging too

A privacy principle and a debugging principle point in the same direction:

> **Share only what is needed for the task.**

Compare:

```text
entire repository
entire notebook
entire patient case
entire traceback
```

with:

```text
small reproducible example
relevant function
tensor shapes
package versions
exact error
expected behaviour
```

The smaller version is often safer, easier for the LLM to understand, easier for you to review, and easier to reproduce.

# 8. "But the model is local"

Running an open-weight model locally can reduce the need to send information to an external service.

But:

> **Local does not automatically mean compliant, secure, or approved.**

Questions still matter:

```text
Is the software approved on this device?
Where are prompts stored?
Are conversation logs written to disk?
Who can access the machine?
Does the software send telemetry?
Are plugins/tools connected to external services?
Where are generated files stored?
Is the workstation approved for this data?
Are backups governed appropriately?
```

Similarly:

```text
open-weight ≠ anonymous
local ≠ approved
enterprise account ≠ permission for every data type
```

# 9. Tool approval and data approval are separate questions

Suppose your institution provides an approved AI assistant.

That is useful, but it does not necessarily mean:

```text
every employee may upload every category of data for every purpose
```

Think of approval as a matrix:

| | Public data | Internal data | Patient data | Secrets |
|---|---:|---:|---:|---:|
| Public consumer AI | ? | No | No! | **No** |
| Institution-approved AI | ? | ? | ? | **No** |
| Local approved environment | ? | ? | ? | **No** |

**You must ask your institution to define them for you.**

# 10. Scenario exercise: would you paste it?

For each case, decide what you would do **before** using an LLM.

### A
Public MSD Spleen data produce a MONAI transform error.

### B
You want an LLM to rewrite an unpublished manuscript Discussion.

### C
A treatment-planning screenshot contains dose, contours, patient ID, and treatment date. An error is visible in the corner.

### D
An API key is hard-coded in your Python script and authentication fails.

### E
Research CTs have been pseudonymised as `Study_001`, `Study_002`, etc. You want to upload one case to a public chatbot.

### F
Your institution provides an approved internal LLM. You want to paste a clinical note into it.

For each scenario ask:

1. What information is involved?
2. Is the tool approved for that information and purpose?
3. Can the problem be solved with public, synthetic, or reduced information?
4. Who should you ask if the policy is unclear?

# 11. If you accidentally shared something sensitive

Do not quietly assume that deleting the prompt solves the problem.

A sensible general response is:

```text
stop further sharing
        ↓
preserve enough information to explain what happened
        ↓
follow the institutional incident/privacy procedure
        ↓
contact the appropriate privacy, security, IT, or data-protection function
        ↓
follow their instructions
```

The exact process differs between institutions.

> **Prompt deletion in a user interface should not be assumed to erase every copy, log, backup, or downstream record.**

Popular AI providers like OpenAI retain data even after deletion. Importantly, also the destinction whether you allow to use your inputs for training in your account data privacy settings, and even whether you gave a feedback (e.g. thumbs up/down) to an AI answer matters.

> **Deleting chats and data is, at best, damage control, at worst you lose the option to trace back your data leakage with the providers.**

Always think **before** you prompt.

# 12. Five questions before you paste

### 1. Is this AI tool approved for this use?

Not only: *Can I access it?*

But also: *Am I approved to do what I am about to do?*

### 2. What kind of information am I about to share?

```text
public?
internal?
personal?
health-related?
confidential?
credential / secret?
```

### 3. Can I solve the problem with less information?

Could you use synthetic data, public data, a code fragment, tensor shapes, or a generic error?

### 4. Where will the information go?

Consider external providers, connected tools, logs, retention, local disk, cloud storage, and service terms.

### 5. Would I be comfortable documenting what I shared and why?

If not, stop and check the policy.

# 13. Why this is part of AI literacy

Responsible AI use is not only about writing good prompts.

It includes understanding:

- what an AI system can and cannot do;
- what information it receives;
- how its use is governed;
- what risks arise in context;
- when human or institutional oversight is required.

In the EU, Article 4 of the AI Act includes an AI-literacy obligation for providers and deployers to support an appropriate level of AI literacy among staff and others operating AI systems on their behalf.

European Commission information:

https://digital-strategy.ec.europa.eu/en/policies/ai-talent-skills-and-literacy

# 14. Exit exercise

You need help debugging a medical-imaging pipeline at work.

Write the sequence you would follow **before opening the LLM**.

A strong answer might be:

```text
1. Identify what information the problem involves.
2. Check whether the AI tool is approved.
3. Check what data classes are permitted.
4. Replace the real patient case with a synthetic example if possible.
5. Reduce code/error to the minimum useful example.
6. Remove credentials, identifiers, paths, and irrelevant metadata.
7. Ask a targeted technical question.
8. Review the response before applying it to real data.
```

# Take-home message

The safest AI coding workflow is not:

```text
problem → paste everything into chatbot → get answer
```

It is:

```text
problem
→ classify the information
→ check policy and tool approval
→ minimise / abstract the input
→ ask a focused question
→ verify the answer
```

> **The LLM may be a powerful coding tool. Access to the tool does not remove your professional, institutional, or legal responsibilities for the information you give it.**

---
## Bridge to the next lessons

In the next lessons we deliberately work in safe teaching environments:

```text
generic or public data + generic code + no patient information
```

There, we use an LLM to start a segmentation project and build a small working AI framework.

Later, we will return to:

- debugging LLM-generated code;
- unnecessary "fluff";
- planner / coder / reviewer workflows.

---
## References and further reading

**European Union**

- Regulation (EU) 2016/679 (GDPR):  
  https://eur-lex.europa.eu/eli/reg/2016/679/oj

- European Data Protection Board — AI and technology:  
  https://www.edpb.europa.eu/topics/ai-and-technology_en

- European Data Protection Board — Anonymisation / pseudonymisation:  
  https://www.edpb.europa.eu/topics/ai-and-technology/anonymisation-pseudonymisation_en

- European Commission — AI literacy:  
  https://digital-strategy.ec.europa.eu/en/policies/ai-talent-skills-and-literacy

**General risk-management context**

- NIST AI Risk Management Framework:  
  https://www.nist.gov/itl/ai-risk-management-framework

Always consult your own institution's current AI policy, acceptable-use policy, privacy/data-protection guidance, information-security policy, research governance, ethics approval, and contracts for approved AI services.